# 第 16 章　予測課題と検証

春には，日合計が近い二日でも，朝と昼で利用が集中する時刻が違うことを確かめました。今度は，前日の終了時点で，翌日の 0〜23 時台それぞれの貸出件数を予測します。まず過去の平均をどの時間帯にも使い，どの時間帯で同じ値を使うと困るのか，後の期間の記録から調べます。

前半は章本文に合わせて上から実行します。すでに書いてある作図のコードは，すべて書き写す必要はありません。後半の約 20 分では，比べる時間帯を自分で選び，日種別ごとに予測の外れ方を調べます。

Colab では「ドライブにコピーを保存」で自分用のノートを作ります。CSV が手元にない場合は公開 URL から読み込みます。ローカルでは，配布 ZIP の配置のまま開くか，ノートと CSV を同じフォルダーに置いて実行できます。

時間別の記録を `hourly` に読み込みます。あとで日付を境に学習用と検証用の行を選ぶため，`parse_dates` で日付を日時型にします。日付の順に，同じ日の中では時刻の順に並べ，先頭の三行を確かめます。

In [ ]:
from pathlib import Path

import pandas as pd

file_name = "bike-sharing-hourly.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

hourly = pd.read_csv(csv_path, parse_dates=["dteday"])
hourly = hourly.sort_values(["dteday", "hr"])
hourly[["dteday", "hr", "workingday", "cnt"]].head(3)

## 本文と一緒に実行

### 一日の形

1 行は，ある日の一つの時間帯を表します。たとえば `hr = 17` の行の `cnt` は，その日の 17 時台に対象地域全体で始まった貸出件数です。

2011 年 6 月 2 日（木）と 11 日（土）は，それぞれ一日の貸出が 4,968 件と 4,966 件でした。二日の線で，どちらの 8 時台が高いか，13 時台に進むとどうなるかを見てください。二日の形を見て，翌日の予測を考えるなら何が手掛かりになりそうでしょうか。

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7.2, 3.6), layout="constrained")
for date, label, color, style, marker in [
    ("2011-06-02", "Jun 2 (Thu)", "#195b8a", "-", "s"),
    ("2011-06-11", "Jun 11 (Sat)", "#a7421f", "--", "o"),
]:
    day = hourly.loc[hourly["dteday"] == date].set_index("hr").reindex(range(24))
    ax.plot(day.index, day["cnt"], label=label, color=color,
            linestyle=style, marker=marker, markersize=3.5)
ax.set(xlabel="Clock hour", ylabel="Rentals per hour",
       xticks=[0, 4, 8, 13, 17, 23], ylim=(0, 650))
ax.legend(frameon=False, loc="upper left")
plt.show()

### 時間順の分割

翌日の曜日・祝日と，何時台を予測するかは，前日の終了時点で分かります。翌日の同じ時間帯に観測された気象や，登録区分別の貸出件数は，その時間帯が来てから分かります。風速の実測値を使って以前の貸出を説明することと，前日終了時の予測に使うことの違いを考えてください。

2011 年の収録行を学習に使い，2012 年 1〜6 月の収録行で予測を採点します。1 月 1 日は検証に入り，7 月 1 日は保留に入ります。保留する 2012 年後半は，秋のモデル比較には使いません。春には全期間の傾向を調べ，後半の一部の日も読んでいるので，この保留は秋の候補比較から切り分けるための区分です。

下の出力で学習・検証の行数を見ます。検証期間の 182 日に 24 の時刻を置くと 4,368 通りありますが，行のない時間帯には実測値がありません。そこで，予測は各時間帯に出せる規則として決め，実測値が収録されている行で採点します。

In [ ]:
date = hourly["dteday"]
valid_start = pd.Timestamp("2012-01-01")
holdout_start = pd.Timestamp("2012-07-01")

train = hourly.loc[date < valid_start]
valid = hourly.loc[(date >= valid_start) & (date < holdout_start)]
period_counts = pd.Series({
    "all": len(hourly),
    "train": len(train),
    "validation": len(valid),
    "held back": (date >= holdout_start).sum(),
}, name="rows")
period_counts

### 一律の平均

どの時間帯かによらず一定の件数を返す規則から始めます。初めのセルの `mean()` には学習期間の貸出件数を渡します。出力の値は一日全体の合計でしょうか。それとも収録された 1 時間当たりの件数でしょうか。

次のセルはその予測値を検証の各行に並べます。2012 年の件数を使って平均を求め直さず，2011 年の平均をそのまま使うと，学習で決めた規則が後の期間でどれだけ当たったかを調べられます。

In [ ]:
baseline = train["cnt"].mean()
print(f"{baseline:.2f}")

In [ ]:
scored = valid.assign(prediction=baseline)
scored[["dteday", "hr", "prediction"]].head(3).round({"prediction": 2})

### 差の向きと大きさ

小例では，2011 年の表とは別の架空の過去 3 行（40，100，160 件）を平均して 100 件と決めたとします。次の日の四つの時間帯が過ぎ，以下の実測値が分かりました。

| 時間帯 | 実測（件） | 予測（件） |
| --- | ---: | ---: |
| 4 時台 | 20 | 100 |
| 6 時台 | 80 | 100 |
| 8 時台 | 150 | 100 |
| 9 時台 | 150 | 100 |

四つの行で `実測 − 予測` を考えます。4 時台は −80 件で予測過大，8 時台は +50 件で予測過小です。この二つの差が平均でどのように打ち消し合うか考えた後，四つの差それぞれの絶対値を取って平均してみてください。差の向きを保つ平均と，外れの大きさを測る平均絶対誤差（MAE）では，表していることが異なります。

実データでも `error` 列を作り，`abs()` で `abs_error` 列を作ります。最初の表示で，2012 年 1 月 1 日の 0 時台の実測と予測を照合し，負の誤差がどちらの値が大きいことを表すかを確認してください。次に，検証の全 4,358 行を平均します。

In [ ]:
scored["error"] = scored["cnt"] - scored["prediction"]
scored["abs_error"] = scored["error"].abs()
result_columns = ["dteday", "hr", "cnt", "prediction", "error", "abs_error"]
scored[result_columns].head(3).round({"prediction": 2, "error": 2, "abs_error": 2})

In [ ]:
mean_error = scored["error"].mean()
mae = scored["abs_error"].mean()
print(f"平均の符号付き誤差: {mean_error:.2f} 件")
print(f"平均絶対誤差 (MAE): {mae:.2f} 件")

### 二日の実測と予測

2012 年 6 月 7 日（木）と 9 日（土）は，春に時間帯による貸出の違いを比べた二日です。今度は両日の実測に基準予測の破線を加えます。二日の縦軸は同じ目盛です。8・13・17 時台では実測と破線の距離を縦線で示します。

どちらの 8 時台で不足が大きいか，土曜日の昼にはどう見えるかを確かめてください。破線を一律に上げたとしたら，昼間の大きな不足と両日の 4 時台の誤差がどう変わるかも考えます。

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, sharex=True, sharey=True, figsize=(7.2, 5.25), layout="constrained")
for ax, date_text, label in zip(
    axes, ["2012-06-07", "2012-06-09"], ["Jun 7 (Thu)", "Jun 9 (Sat)"]
):
    day = scored.loc[scored["dteday"] == date_text].set_index("hr").reindex(range(24))
    ax.plot(day.index, day["cnt"], color="#195b8a", marker="o", markersize=3.4,
            label="Observed")
    ax.axhline(baseline, color="#343a42", linestyle="--", label="Constant forecast")
    selected = day.loc[[8, 13, 17]]
    ax.vlines(selected.index, selected["prediction"], selected["cnt"],
              color="#a7421f", linestyle=":", linewidth=1.8)
    ax.set(title=label, ylabel="Rentals per hour", ylim=(0, 960))
axes[0].legend(frameon=False, loc="upper left")
axes[-1].set(xlabel="Clock hour", xticks=[0, 4, 8, 13, 17, 23])
plt.show()

### 時間帯ごとの誤差

二日の例を，検証期間の半年へ広げます。同じ時刻の行を `groupby("hr")` で集め，符号付きの平均誤差と MAE を求めます。集計を変えても，もとの予測値は 2011 年の平均のままです。

上の図は誤差の向き，下の図は外れの大きさを表します。例えば上の 22 時台は 0 に近い棒です。同じ 22 時台の下の棒を見て，各日の外れが少ないと言えるか考えてください。

In [ ]:
by_hour = scored.groupby("hr").agg(
    rows=("cnt", "size"),
    mean_error=("error", "mean"),
    mae=("abs_error", "mean"),
)
by_hour.loc[[4, 8, 13, 17, 22]].round(2)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, sharex=True, figsize=(7.2, 5.15), layout="constrained")
colors = by_hour["mean_error"].ge(0).map({True: "#a7421f", False: "#195b8a"})
axes[0].bar(by_hour.index, by_hour["mean_error"], color=colors)
axes[0].axhline(0, color="#343a42", linewidth=0.8)
axes[0].set(title="Mean signed error", ylabel="Rentals per hour",
            yticks=[-150, 0, 150, 300, 450], ylim=(-170, 475))
axes[1].bar(by_hour.index, by_hour["mae"], color="#5a6d7b")
axes[1].axhline(mae, color="#343a42", linestyle="--", label="Overall MAE")
axes[1].set(title="Mean absolute error", xlabel="Clock hour",
            ylabel="Rentals per hour", xticks=[0, 4, 8, 13, 17, 23],
            yticks=[0, 150, 300, 450], ylim=(0, 475))
axes[1].legend(frameon=False, loc="upper left")
plt.show()

### 月による違い

同じ予測を半年使うと，よく当たる月と外れが大きい月があるでしょうか。`dteday` から月を取り出し，月別の実測の平均と MAE を並べます。

どの月でも一律の予測を使っています。実測の平均が破線に近い月についても，MAE を読んでください。月の平均をうまく見積もることと，その月の各時間帯をうまく見積もることが同じかを考えます。

In [ ]:
month = scored["dteday"].dt.month.rename("month")
by_month = scored.groupby(month).agg(
    rows=("cnt", "size"),
    mean_observed=("cnt", "mean"),
    mae=("abs_error", "mean"),
)
by_month.round(2)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7.2, 3.6), layout="constrained")
ax.plot(by_month.index, by_month["mean_observed"], color="#195b8a",
        marker="o", label="Mean observed")
ax.plot(by_month.index, by_month["mae"], color="#a7421f",
        marker="s", linestyle=":", label="MAE")
ax.axhline(baseline, color="#343a42", linestyle="--", label="Constant forecast")
ax.set(xlabel="Month (2012)", ylabel="Rentals per hour",
       xticks=by_month.index, ylim=(0, 320))
ax.legend(frameon=False, loc="upper left")
plt.show()

## 実習（約 20 分）

時間帯を一つ選び，**土日・祝日とそれ以外の日で，一律予測の外れ方が変わるか**を調べます。例えば，朝の件数を予測する規則に日種別を加える意味を考えるなら，朝だけを取り出し，同じ時間帯の中で二つの日種別を比べます。`workingday = 0` は土日・祝日，`workingday = 1` はそれ以外の日です。

### 1. 選んで予想する（4 分）

早朝（1〜4 時台），朝（7〜9 時台），昼（12〜14 時台），夕方（16〜18 時台），夜（20〜22 時台）から一つ選びます。集計する前に，その時間帯を選んだ理由と，どちらの日種別で予測が多すぎる，または足りなくなると思うかを書きます。二日分の線や半年の棒のどこから予想したかも添えてください。

**選んだ時間帯と予想**　…

### 2. 日種別で比べる（8 分）

次のセルの `None` を，選んだ時間帯の最初と最後の数を入れた組に替えます。例えば 10〜11 時台を指定する書き方は `(10, 11)` です。`between()` は両端を含めて，その時間帯の行だけを選びます。

結果の二つの行で，`rows` は採点した行数，`mean_error` は符号付き平均誤差，`mae` は MAE です。まず二つの日種別が入っていることを確かめます。次に符号から多すぎるか足りないかを読み，MAE から外れの大きさを比べます。行数の差は何を数えた差でしょうか。

In [ ]:
hour_range = None  # (最初の時刻, 最後の時刻) に置き換える

if hour_range is None:
    print("選んだ時間帯の最初と最後を hour_range に指定してください。")
else:
    start_hour, end_hour = hour_range
    study_hours = scored.loc[scored["hr"].between(start_hour, end_hour)]
    study_by_day_type = study_hours.groupby("workingday").agg(
        rows=("cnt", "size"),
        mean_error=("error", "mean"),
        mae=("abs_error", "mean"),
    )
    display(study_by_day_type.round(2))

### 3. 次に調べたいことを書く（8 分）

2012 年 1〜6 月の収録行について，選んだ時間帯でどちらの日種別がどの向きに外れ，どちらの外れが大きかったかを，表の誤差と行数を使って 2〜3 文で書きます。符号付きの平均誤差と MAE が違うことが読み取れるよう，結果の名前も添えます。最初の予想を保つ部分や修正する部分を示して，次の予測に加えて確かめたい情報を一つ挙げてください。その情報が前日の終了時点で分かるかも説明します。

**結果と次の予測に使いたい情報**　…

余裕があれば，選んだ時間帯での日種別の違いは他の時間帯でも見えると思うか，予想してから時間帯を一度替えます。同じ日種別でも時間帯を変えると外れ方が変わるか，最初の表と照らしてください。

提出する場合は，報告に使った時間帯をセルに指定し，カーネルを再起動して上から全セルを実行します。選んだ時間帯，集計表，短い説明が残る状態でノートを保存してください。

### データの出典

Capital Bikeshare の時間別集計は，Hadi Fanaee-T が UCI Machine Learning Repository で公開した `hour.csv` のファイル名と改行コードを変更したものです。 [データ DOI](https://doi.org/10.24432/C5W894) ・ [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)